# Group 2 — Long-Document Understanding Evaluation

This notebook benchmarks **5 memory-augmented AI systems** on three long-document comprehension datasets, each with 250 questions.

## What this notebook does
1. Downloads the three evaluation datasets from HuggingFace.
2. For each of the 5 systems, runs an evaluation loop over all three datasets and saves results incrementally.
3. Loads saved results and computes the appropriate metric for each dataset.
4. Prints a summary comparison table.

## Systems evaluated
| System | Approach | Python env |
|--------|----------|-----------|
| **ReadAgent** | 3-phase: paginate → gist → lookup & answer | Python 3.13 |
| **A-Mem** | Agentic memory graph with keyword-expansion retrieval | Python 3.13 |
| **MemTree** | Hierarchical semantic memory tree | Python 3.13 |
| **MemGPT** | Tiered memory OS (working / recall / archival) | Python 3.10+ |
| **MemOS** | Cloud-hosted structured memory via MemOS API | Python 3.13 |

## Datasets & metrics
| Dataset | Task | Metric | Notes |
|---------|------|--------|-------|
| **QuALITY** | MCQ on long sci-fi articles (~30 K chars) | Accuracy % | gold_label is 1-indexed int (1=A … 4=D) |
| **NarrativeQA** | Open QA on story summaries | Token F1 % | token overlap vs best of answer1/answer2 |
| **HaluEval** | Factual QA with hallucination trap | Contains-Match % | right_answer.lower() in model_answer.lower() |

## Result files
All results are saved incrementally to `../results/group2/` as JSON files.

> **Python environment note:** Systems that require `sentence-transformers` (ReadAgent, A-Mem, MemTree, MemOS) **must** be run with **Python 3.13** due to a NumPy 2.x / PyTorch 1.x incompatibility in older environments. MemGPT works with Python 3.10+.


---
## 0 — Setup
Install dependencies, load environment variables, and define shared metric utilities.

In [ ]:
# Install common dependencies (run once)
# %pip install openai python-dotenv huggingface_hub pandas tabulate

In [ ]:
import os
import json
import re
import sys
import time
import pathlib
import string
from collections import Counter
from typing import List, Dict, Any, Optional, Tuple

# Load .env for API keys
try:
    from dotenv import load_dotenv
    load_dotenv(dotenv_path=pathlib.Path("../.env"), override=False)
    print("[info] .env loaded")
except ImportError:
    print("[warn] python-dotenv not installed; relying on shell environment")

OPENAI_API_KEY = os.environ.get("OPENAI_API_KEY", "")
MEMOS_API_KEY  = os.environ.get("MEMOS_API_KEY", "")   # for MemOS cloud API
if not OPENAI_API_KEY:
    print("[warn] OPENAI_API_KEY not set")

# ── Directory layout ──────────────────────────────────────────────────────────
NOTEBOOK_DIR = pathlib.Path(".").resolve()
REPO_ROOT    = NOTEBOOK_DIR.parent
DATA_DIR     = REPO_ROOT / "data" / "group2"
RESULTS_DIR  = REPO_ROOT / "results" / "group2"

DATA_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print(f"[info] Data dir   : {DATA_DIR}")
print(f"[info] Results dir: {RESULTS_DIR}")

# ── Metric helpers ────────────────────────────────────────────────────────────

def accuracy_mcq(records: List[Dict], pred_key: str, gold_key: str) -> Dict[str, Any]:
    """
    MCQ accuracy for QuALITY.
    pred_key  : key holding the model's predicted choice as int 1-4 (or letter A-D).
    gold_key  : key holding gold_label as int 1-4.
    """
    letter_map = {"A": 1, "B": 2, "C": 3, "D": 4,
                  "a": 1, "b": 2, "c": 3, "d": 4}
    correct = total = 0
    for r in records:
        pred = r.get(pred_key)
        gold = r.get(gold_key)
        if pred is None or gold is None:
            continue
        # Normalise pred to int
        if isinstance(pred, str):
            pred = letter_map.get(pred.strip(), None)
            if pred is None:
                continue
        try:
            pred = int(pred)
            gold = int(gold)
        except (ValueError, TypeError):
            continue
        total += 1
        if pred == gold:
            correct += 1
    acc = round(100 * correct / total, 1) if total > 0 else 0.0
    return {"correct": correct, "total": total, "accuracy_pct": acc}


def _tokenize(text: str) -> List[str]:
    """Lowercase, strip punctuation, split on whitespace."""
    text = text.lower()
    text = text.translate(str.maketrans("", "", string.punctuation))
    return text.split()


def token_f1_single(prediction: str, reference: str) -> float:
    """Compute token-level F1 between prediction and reference."""
    pred_tokens = _tokenize(prediction)
    ref_tokens  = _tokenize(reference)
    if not pred_tokens or not ref_tokens:
        return 0.0
    common = Counter(pred_tokens) & Counter(ref_tokens)
    num_common = sum(common.values())
    if num_common == 0:
        return 0.0
    precision = num_common / len(pred_tokens)
    recall    = num_common / len(ref_tokens)
    f1 = 2 * precision * recall / (precision + recall)
    return f1


def token_f1_best(prediction: str, ref1: str, ref2: str) -> float:
    """Return max token F1 against either reference answer."""
    return max(token_f1_single(prediction, ref1),
               token_f1_single(prediction, ref2))


def score_narrativeqa(records: List[Dict],
                      pred_key: str = "model_answer",
                      ref1_key: str = "reference_answer1",
                      ref2_key: str = "reference_answer2") -> Dict[str, Any]:
    """Compute mean token F1 over all NarrativeQA records."""
    scores = []
    for r in records:
        pred = str(r.get(pred_key, ""))
        ref1 = str(r.get(ref1_key, ""))
        ref2 = str(r.get(ref2_key, ""))
        scores.append(token_f1_best(pred, ref1, ref2))
    mean_f1 = round(100 * sum(scores) / len(scores), 1) if scores else 0.0
    return {"total": len(scores), "mean_token_f1_pct": mean_f1}


def contains_match(prediction: str, ground_truth: str) -> bool:
    """Case-insensitive substring match (HaluEval metric)."""
    if not ground_truth or not ground_truth.strip():
        return False
    return ground_truth.strip().lower() in str(prediction).lower()


def score_halueval(records: List[Dict],
                   pred_key: str = "model_answer",
                   gt_key: str   = "right_answer") -> Dict[str, Any]:
    """Compute contains-match accuracy over HaluEval records."""
    correct = total = 0
    for r in records:
        gt   = str(r.get(gt_key, ""))
        pred = str(r.get(pred_key, ""))
        if not gt.strip():
            continue
        total += 1
        if contains_match(pred, gt):
            correct += 1
    acc = round(100 * correct / total, 1) if total > 0 else 0.0
    return {"correct": correct, "total": total, "accuracy_pct": acc}


DATASETS = ["quality", "narrativeqa", "halueval"]
print("[info] Setup complete.")


---
## 1 — Download Datasets

Downloads the three Group 2 evaluation datasets from HuggingFace:
`darshan3j/memory-systems-eval-datasets`

Files saved to `../data/group2/`.

| Dataset | File | Records |
|---------|------|---------|
| QuALITY | `group2/quality_250_filtered.json` | 250 MCQ questions |
| NarrativeQA | `group2/narrativeqa_250_filtered.json` | 250 open QA pairs |
| HaluEval | `group2/halueval_250_filtered.json` | 250 factual QA pairs |


In [ ]:
# %pip install huggingface_hub

from huggingface_hub import hf_hub_download
import shutil

HF_REPO  = "darshan3j/memory-systems-eval-datasets"
HF_FILES = {
    "quality"     : "group2/quality_250_filtered.json",
    "narrativeqa" : "group2/narrativeqa_250_filtered.json",
    "halueval"    : "group2/halueval_250_filtered.json",
}

dataset_paths: Dict[str, pathlib.Path] = {}

for name, hf_path in HF_FILES.items():
    local_path = DATA_DIR / f"{name}_250_filtered.json"
    if local_path.exists():
        print(f"[skip] {name} already downloaded -> {local_path}")
    else:
        print(f"[download] {name} ...")
        downloaded = hf_hub_download(
            repo_id   = HF_REPO,
            filename  = hf_path,
            repo_type = "dataset",
            local_dir = str(DATA_DIR),
        )
        src = pathlib.Path(downloaded)
        if src != local_path:
            shutil.copy(src, local_path)
        print(f"[ok]   {name} -> {local_path}")
    dataset_paths[name] = local_path

# Sanity check
for name, path in dataset_paths.items():
    data = json.loads(path.read_text(encoding="utf-8"))
    print(f"  {name}: {len(data)} records")


In [ ]:
def load_dataset(name: str) -> List[Dict]:
    """Load a Group-2 dataset by name (quality / narrativeqa / halueval)."""
    path = dataset_paths[name]
    return json.loads(path.read_text(encoding="utf-8"))


# ---- Dataset field reference ------------------------------------------------
# QuALITY    : {article_id, title, author, topic, article, question, options,
#               gold_label (1-indexed int), question_unique_id, difficult}
# NarrativeQA: {document_id, summary, question, answer1, answer2}
# HaluEval   : {knowledge, question, right_answer, hallucinated_answer}

# Preview one record per dataset
for name in DATASETS:
    sample = load_dataset(name)[0]
    print(f"\n{name} sample keys: {list(sample.keys())}")
    for k, v in sample.items():
        sv = str(v)
        suffix = "..." if len(sv) > 120 else ""
        print(f"  {k}: {sv[:120]}{suffix}")


---
## 2 — ReadAgent

**Description:** ReadAgent (Laban et al., 2023) is a 3-phase pipeline for long-document comprehension:

1. **Episode Pagination** — split the article into pages of ~300 words
2. **Memory Gisting** — compress each page into a short gist sentence (working memory)
3. **Look-Up & Answer** — identify which pages are relevant from gists, retrieve full text, answer

**Paper:** [ReadAgent: A Continuous Memory Narrative for Long-Context QA (Laban et al., 2023)](https://arxiv.org/abs/2309.11206)

**Python env:** Python 3.13 (only `openai` required — no sentence-transformers).

**Per-dataset strategy:**
- **QuALITY:** paginate full article (~30K chars) → gist pages → lookup → answer MCQ (A/B/C/D)
- **NarrativeQA:** same 3-phase pipeline on shorter story summary text
- **HaluEval:** passages are short (~350 chars), answered directly in a single LLM call

**Result files:** `../results/group2/readagent_{dataset}_250_results.json`


In [ ]:
# %pip install openai

# ---- ReadAgent system setup -------------------------------------------------
import openai

READAGENT_MODEL  = "gpt-4o-mini"  # model used for all 3 ReadAgent phases
WORDS_PER_PAGE   = 300            # episode pagination page size (words)
MAX_LOOKUP_PAGES = 3              # max pages to retrieve in look-up phase

_ra_client = openai.OpenAI(api_key=OPENAI_API_KEY)


def _chat(messages: List[Dict], model: str = READAGENT_MODEL,
          max_tokens: int = 512, temperature: float = 0.0) -> str:
    """Single OpenAI chat completion, returns content string."""
    resp = _ra_client.chat.completions.create(
        model=model, messages=messages,
        max_tokens=max_tokens, temperature=temperature,
    )
    return resp.choices[0].message.content.strip()


def paginate_text(text: str, words_per_page: int = WORDS_PER_PAGE) -> List[str]:
    """Split text into pages of approximately words_per_page words."""
    words = text.split()
    return [" ".join(words[i:i+words_per_page]) for i in range(0, len(words), words_per_page)]


def gist_pages(pages: List[str]) -> List[str]:
    """Phase 2: compress each page to a one-sentence gist via LLM."""
    gists = []
    for page in pages:
        msg = [
            {"role": "system", "content": "Summarise the passage in one concise sentence."},
            {"role": "user",   "content": page},
        ]
        gists.append(_chat(msg, max_tokens=80))
    return gists


def lookup_pages(
    gists: List[str], pages: List[str], question: str, k: int = MAX_LOOKUP_PAGES
) -> Tuple[List[str], List[int]]:
    """Phase 3a: ask LLM which pages are most relevant; return (texts, 0-based indices)."""
    numbered = "\n".join(f"Page {i+1}: {g}" for i, g in enumerate(gists))
    system_prompt = (
        "You are given page summaries of a document and a question. "
        f"List the numbers (1-indexed) of up to {k} pages most relevant to answer "
        "the question. Reply ONLY with comma-separated page numbers, e.g. 2,5,7"
    )
    msg = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": f"Page summaries:\n{numbered}\n\nQuestion: {question}"},
    ]
    raw = _chat(msg, max_tokens=32)
    indices = []
    for tok in re.split(r"[,\s]+", raw):
        try:
            idx = int(tok.strip()) - 1
            if 0 <= idx < len(pages):
                indices.append(idx)
        except ValueError:
            pass
    indices = sorted(set(indices))[:k] or [0]  # fallback to first page
    return [pages[i] for i in indices], indices


def readagent_answer_mcq(
    article: str, question: str, options: List[str]
) -> Tuple[str, int, List[int], int]:
    """Full ReadAgent pipeline for MCQ. Returns (raw, choice_int, lu_indices, word_count)."""
    word_count = len(article.split())
    pages = paginate_text(article)
    if len(pages) > 1:
        gists = gist_pages(pages)
        retrieved, lu_idx = lookup_pages(gists, pages, question)
    else:
        retrieved, lu_idx = pages, [0]
    context  = "\n\n".join(retrieved)
    opts_str = "\n".join(f"{chr(64+i+1)}. {o}" for i, o in enumerate(options))
    system_msg = ("Answer the multiple-choice question using the context. "
                  "Reply with ONLY the letter A, B, C, or D.")
    msg = [
        {"role": "system", "content": system_msg},
        {"role": "user",   "content": f"Context:\n{context}\n\nQuestion: {question}\n\nOptions:\n{opts_str}"},
    ]
    raw = _chat(msg, max_tokens=8)
    choice_int = {"A": 1, "B": 2, "C": 3, "D": 4}.get(raw.strip().upper()[:1], 0)
    return raw, choice_int, lu_idx, word_count


def readagent_answer_open(text: str, question: str) -> Tuple[str, List[int], int]:
    """Full ReadAgent pipeline for open-ended QA. Returns (answer, lu_indices, word_count)."""
    word_count = len(text.split())
    pages = paginate_text(text)
    if len(pages) > 1:
        gists = gist_pages(pages)
        retrieved, lu_idx = lookup_pages(gists, pages, question)
    else:
        retrieved, lu_idx = pages, [0]
    context = "\n\n".join(retrieved)
    msg = [
        {"role": "system", "content": "Answer the question concisely based on the context."},
        {"role": "user",   "content": f"Context:\n{context}\n\nQuestion: {question}"},
    ]
    return _chat(msg, max_tokens=256), lu_idx, word_count


print("[ok] ReadAgent helpers defined.")


In [ ]:
# ---- ReadAgent evaluation loop ------------------------------------------
# Saves results incrementally every 10 questions.
# Resume-safe: already-processed questions are skipped.

def run_readagent_quality(force: bool = False):
    out_path = RESULTS_DIR / "readagent_quality_250_results.json"
    results: List[Dict] = []
    done_ids: set = set()
    if out_path.exists() and not force:
        results  = json.loads(out_path.read_text(encoding="utf-8"))
        done_ids = {r["question_unique_id"] for r in results}
        print(f"[resume] quality: {len(done_ids)} already done")
    data      = load_dataset("quality")
    remaining = [q for q in data if q["question_unique_id"] not in done_ids]
    if not remaining:
        print("[skip] quality: already complete"); return
    print(f"[run] quality: {len(remaining)} questions remaining ...")
    for i, item in enumerate(remaining, 1):
        qid = item["question_unique_id"]
        try:
            raw, choice_int, lu_idx, wc = readagent_answer_mcq(
                item["article"], item["question"], item["options"])
            gold = int(item["gold_label"])
            is_correct = (choice_int == gold)
        except Exception as e:
            print(f"  [err] {qid}: {e}")
            raw, choice_int, lu_idx, wc = "", 0, [], 0
            gold = int(item.get("gold_label", 0))
            is_correct = False
        results.append({
            "question_unique_id": qid,
            "article_id"        : item.get("article_id", ""),
            "title"             : item.get("title", ""),
            "question"          : item["question"],
            "options"           : item["options"],
            "gold_label"        : gold,
            "model_choice"      : choice_int,
            "is_correct"        : is_correct,
            "raw_response"      : raw,
            "pages_looked_up"   : lu_idx,
            "word_count"        : wc,
        })
        if i % 10 == 0 or i == len(remaining):
            out_path.write_text(json.dumps(results, indent=2, ensure_ascii=False), encoding="utf-8")
            print(f"  [{i}/{len(remaining)}] saved")
    print(f"[done] quality -> {out_path}")


def run_readagent_narrativeqa(force: bool = False):
    out_path = RESULTS_DIR / "readagent_narrativeqa_250_results.json"
    results: List[Dict] = []
    done_ids: set = set()
    if out_path.exists() and not force:
        results  = json.loads(out_path.read_text(encoding="utf-8"))
        done_ids = {r["document_id"] for r in results}
        print(f"[resume] narrativeqa: {len(done_ids)} already done")
    data      = load_dataset("narrativeqa")
    remaining = [q for q in data if q["document_id"] not in done_ids]
    if not remaining:
        print("[skip] narrativeqa: already complete"); return
    print(f"[run] narrativeqa: {len(remaining)} questions remaining ...")
    for i, item in enumerate(remaining, 1):
        did = item["document_id"]
        try:
            answer, lu_idx, wc = readagent_answer_open(item["summary"], item["question"])
        except Exception as e:
            print(f"  [err] {did}: {e}")
            answer, lu_idx, wc = "", [], 0
        results.append({
            "document_id"      : did,
            "question"         : item["question"],
            "reference_answer1": item["answer1"],
            "reference_answer2": item["answer2"],
            "model_answer"     : answer,
            "pages_looked_up"  : lu_idx,
            "word_count"       : wc,
        })
        if i % 10 == 0 or i == len(remaining):
            out_path.write_text(json.dumps(results, indent=2, ensure_ascii=False), encoding="utf-8")
            print(f"  [{i}/{len(remaining)}] saved")
    print(f"[done] narrativeqa -> {out_path}")


def run_readagent_halueval(force: bool = False):
    out_path = RESULTS_DIR / "readagent_halueval_250_results.json"
    results: List[Dict] = []
    done_qs: set = set()
    if out_path.exists() and not force:
        results = json.loads(out_path.read_text(encoding="utf-8"))
        done_qs = {r["question"] for r in results}
        print(f"[resume] halueval: {len(done_qs)} already done")
    data      = load_dataset("halueval")
    remaining = [q for q in data if q["question"] not in done_qs]
    if not remaining:
        print("[skip] halueval: already complete"); return
    print(f"[run] halueval: {len(remaining)} questions remaining ...")
    for i, item in enumerate(remaining, 1):
        question = item["question"]
        try:
            # HaluEval passages are short; answered directly without gisting
            answer, lu_idx, wc = readagent_answer_open(item["knowledge"], question)
        except Exception as e:
            print(f"  [err] q{i}: {e}")
            answer, lu_idx, wc = "", [], 0
        results.append({
            "question"            : question,
            "knowledge"           : item["knowledge"],
            "right_answer"        : item["right_answer"],
            "hallucinated_answer" : item["hallucinated_answer"],
            "model_answer"        : answer,
            "pages_looked_up"     : lu_idx,
            "word_count"          : wc,
        })
        if i % 10 == 0 or i == len(remaining):
            out_path.write_text(json.dumps(results, indent=2, ensure_ascii=False), encoding="utf-8")
            print(f"  [{i}/{len(remaining)}] saved")
    print(f"[done] halueval -> {out_path}")


# Un-comment to run:
# run_readagent_quality()
# run_readagent_narrativeqa()
# run_readagent_halueval()


In [ ]:
# ---- ReadAgent: load results & compute scores ---------------------------
readagent_scores: Dict[str, Any] = {}

# QuALITY (Accuracy %)
_path = RESULTS_DIR / "readagent_quality_250_results.json"
if _path.exists():
    _rec = json.loads(_path.read_text(encoding="utf-8"))
    readagent_scores["quality"] = accuracy_mcq(_rec, pred_key="model_choice", gold_key="gold_label")
    _r = readagent_scores["quality"]
    print(f"  ReadAgent | quality     | {_r['correct']:3d}/{_r['total']:3d} = {_r['accuracy_pct']:5.1f}%")
else:
    print("[missing] readagent quality -- run the evaluation loop first")
    readagent_scores["quality"] = None

# NarrativeQA (Token F1 %)
_path = RESULTS_DIR / "readagent_narrativeqa_250_results.json"
if _path.exists():
    _rec = json.loads(_path.read_text(encoding="utf-8"))
    readagent_scores["narrativeqa"] = score_narrativeqa(_rec)
    _r = readagent_scores["narrativeqa"]
    print(f"  ReadAgent | narrativeqa | {_r['total']:3d} records, mean token F1 = {_r['mean_token_f1_pct']:5.1f}%")
else:
    print("[missing] readagent narrativeqa -- run the evaluation loop first")
    readagent_scores["narrativeqa"] = None

# HaluEval (Contains-Match %)
_path = RESULTS_DIR / "readagent_halueval_250_results.json"
if _path.exists():
    _rec = json.loads(_path.read_text(encoding="utf-8"))
    readagent_scores["halueval"] = score_halueval(_rec)
    _r = readagent_scores["halueval"]
    print(f"  ReadAgent | halueval    | {_r['correct']:3d}/{_r['total']:3d} = {_r['accuracy_pct']:5.1f}%")
else:
    print("[missing] readagent halueval -- run the evaluation loop first")
    readagent_scores["halueval"] = None


---
## 3 — A-Mem

**Description:** A-Mem (Agentic Memory) chunks documents into overlapping segments, stores each chunk as a node in a memory graph using LLM-generated keywords and semantic embeddings, then retrieves relevant nodes at query time via keyword expansion.

**GitHub:** [agiresearch/A-MEM](https://github.com/agiresearch/A-MEM)

**Python env:** Python 3.13 required (`sentence-transformers` needs NumPy 2.x; pyenv 3.10.1 has a NumPy/torch incompatibility that causes `model.encode()` to fail).

**Chunking:** 500 words per chunk for QuALITY and NarrativeQA; 300 words for HaluEval.

**Important:** Wrap `agent.add_memory(chunk)` in try/except — can raise HTTP 400 errors on special characters. These chunks are safely skipped.

**Result files:** `../results/group2/amem_{dataset}_250_results.json`


In [ ]:
# %pip install sentence-transformers openai  (use Python 3.13)

# ---- A-Mem system setup -------------------------------------------------
# A-Mem requires the A-MEM repo to be available. Set AMEM_ROOT to the repo
# directory and the code below will add it to sys.path.

AMEM_ROOT = os.environ.get("AMEM_ROOT", "")  # e.g. C:/path/to/A-MEM
if AMEM_ROOT and pathlib.Path(AMEM_ROOT).exists():
    sys.path.insert(0, AMEM_ROOT)
    print(f"[ok] A-Mem root added: {AMEM_ROOT}")
else:
    print("[warn] AMEM_ROOT not set or not found; A-Mem import may fail")

try:
    from agenticMemory import MemoryAgent  # type: ignore
    print("[ok] A-Mem MemoryAgent imported")
except ImportError as e:
    print(f"[error] Could not import MemoryAgent: {e}")

AMEM_CHUNK_SIZES = {
    "quality"    : 500,  # words per chunk
    "narrativeqa": 500,
    "halueval"   : 300,
}
AMEM_LLM_MODEL = "gpt-4o-mini"


def chunk_text(text: str, chunk_size: int) -> List[str]:
    """Split text into chunks of chunk_size words."""
    words = text.split()
    return [" ".join(words[i:i+chunk_size]) for i in range(0, len(words), chunk_size)]


def amem_answer_question(
    agent,
    text: str,
    question: str,
    chunk_size: int,
    system_prompt: str = "Answer the question based on the retrieved memories.",
) -> Tuple[str, int]:
    """
    Load text into A-Mem then retrieve and answer the question.
    Returns (model_answer, num_chunks_added).
    """
    chunks = chunk_text(text, chunk_size)
    added  = 0
    for chunk in chunks:
        try:
            agent.add_memory(chunk)
            added += 1
        except Exception:
            pass  # skip chunks that cause API errors (special chars, etc.)
    try:
        response = agent.answer_question(question, system_prompt=system_prompt)
    except Exception as e:
        response = f"[error] {e}"
    return str(response), added


def _make_amem_agent():
    """Create a new A-Mem MemoryAgent with OpenAI backend."""
    return MemoryAgent(model=AMEM_LLM_MODEL, api_key=OPENAI_API_KEY)


print("[ok] A-Mem helpers defined.")


In [ ]:
# ---- A-Mem evaluation loop ----------------------------------------------
# Each question gets a FRESH MemoryAgent instance so memories from one
# document do not bleed into another question.

def run_amem_quality(force: bool = False):
    out_path = RESULTS_DIR / "amem_quality_250_results.json"
    results: List[Dict] = []
    done_ids: set = set()
    if out_path.exists() and not force:
        results  = json.loads(out_path.read_text(encoding="utf-8"))
        done_ids = {r["question_unique_id"] for r in results}
        print(f"[resume] quality: {len(done_ids)} already done")
    data      = load_dataset("quality")
    remaining = [q for q in data if q["question_unique_id"] not in done_ids]
    if not remaining:
        print("[skip] quality: already complete"); return
    print(f"[run] quality: {len(remaining)} questions remaining ...")
    chunk_size = AMEM_CHUNK_SIZES["quality"]
    for i, item in enumerate(remaining, 1):
        qid = item["question_unique_id"]
        try:
            agent = _make_amem_agent()
            opts_str = "  ".join(f"{chr(64+j+1)}. {o}" for j, o in enumerate(item["options"]))
            system_p = ("Answer the multiple-choice question using the retrieved memories. "
                        "Reply with ONLY the letter A, B, C, or D.")
            full_q = f"{item['question']}\n\nOptions:\n{opts_str}"
            raw_answer, num_chunks = amem_answer_question(
                agent, item["article"], full_q, chunk_size, system_prompt=system_p)
            choice_int = {"A":1,"B":2,"C":3,"D":4}.get(raw_answer.strip().upper()[:1], 0)
            gold = int(item["gold_label"])
            is_correct = (choice_int == gold)
        except Exception as e:
            print(f"  [err] {qid}: {e}")
            raw_answer, choice_int, num_chunks = f"[error] {e}", 0, 0
            gold = int(item.get("gold_label", 0))
            is_correct = False
        results.append({
            "question_unique_id": qid,
            "article_id"        : item.get("article_id", ""),
            "title"             : item.get("title", ""),
            "question"          : item["question"],
            "options"           : item["options"],
            "gold_label"        : gold,
            "model_choice"      : choice_int,
            "is_correct"        : is_correct,
            "raw_answer"        : raw_answer,
            "num_chunks"        : num_chunks,
        })
        if i % 10 == 0 or i == len(remaining):
            out_path.write_text(json.dumps(results, indent=2, ensure_ascii=False), encoding="utf-8")
            print(f"  [{i}/{len(remaining)}] saved")
    print(f"[done] quality -> {out_path}")


def run_amem_narrativeqa(force: bool = False):
    out_path = RESULTS_DIR / "amem_narrativeqa_250_results.json"
    results: List[Dict] = []
    done_ids: set = set()
    if out_path.exists() and not force:
        results  = json.loads(out_path.read_text(encoding="utf-8"))
        done_ids = {r["document_id"] for r in results}
        print(f"[resume] narrativeqa: {len(done_ids)} already done")
    data      = load_dataset("narrativeqa")
    remaining = [q for q in data if q["document_id"] not in done_ids]
    if not remaining:
        print("[skip] narrativeqa: already complete"); return
    print(f"[run] narrativeqa: {len(remaining)} questions remaining ...")
    chunk_size = AMEM_CHUNK_SIZES["narrativeqa"]
    for i, item in enumerate(remaining, 1):
        did = item["document_id"]
        try:
            agent = _make_amem_agent()
            model_answer, num_chunks = amem_answer_question(
                agent, item["summary"], item["question"], chunk_size)
        except Exception as e:
            print(f"  [err] {did}: {e}")
            model_answer, num_chunks = f"[error] {e}", 0
        results.append({
            "document_id"      : did,
            "question"         : item["question"],
            "reference_answer1": item["answer1"],
            "reference_answer2": item["answer2"],
            "model_answer"     : model_answer,
            "num_chunks"       : num_chunks,
        })
        if i % 10 == 0 or i == len(remaining):
            out_path.write_text(json.dumps(results, indent=2, ensure_ascii=False), encoding="utf-8")
            print(f"  [{i}/{len(remaining)}] saved")
    print(f"[done] narrativeqa -> {out_path}")


def run_amem_halueval(force: bool = False):
    out_path = RESULTS_DIR / "amem_halueval_250_results.json"
    results: List[Dict] = []
    done_qs: set = set()
    if out_path.exists() and not force:
        results = json.loads(out_path.read_text(encoding="utf-8"))
        done_qs = {r["question"] for r in results}
        print(f"[resume] halueval: {len(done_qs)} already done")
    data      = load_dataset("halueval")
    remaining = [q for q in data if q["question"] not in done_qs]
    if not remaining:
        print("[skip] halueval: already complete"); return
    print(f"[run] halueval: {len(remaining)} questions remaining ...")
    chunk_size = AMEM_CHUNK_SIZES["halueval"]
    for i, item in enumerate(remaining, 1):
        question = item["question"]
        try:
            agent = _make_amem_agent()
            model_answer, num_chunks = amem_answer_question(
                agent, item["knowledge"], question, chunk_size)
        except Exception as e:
            print(f"  [err] q{i}: {e}")
            model_answer, num_chunks = f"[error] {e}", 0
        results.append({
            "question"            : question,
            "knowledge"           : item["knowledge"],
            "right_answer"        : item["right_answer"],
            "hallucinated_answer" : item["hallucinated_answer"],
            "model_answer"        : model_answer,
            "num_chunks"          : num_chunks,
        })
        if i % 10 == 0 or i == len(remaining):
            out_path.write_text(json.dumps(results, indent=2, ensure_ascii=False), encoding="utf-8")
            print(f"  [{i}/{len(remaining)}] saved")
    print(f"[done] halueval -> {out_path}")


# Un-comment to run:
# run_amem_quality()
# run_amem_narrativeqa()
# run_amem_halueval()


In [ ]:
# ---- A-Mem: load results & compute scores -------------------------------
amem_scores: Dict[str, Any] = {}

# QuALITY
_path = RESULTS_DIR / "amem_quality_250_results.json"
if _path.exists():
    _rec = json.loads(_path.read_text(encoding="utf-8"))
    amem_scores["quality"] = accuracy_mcq(_rec, pred_key="model_choice", gold_key="gold_label")
    _r = amem_scores["quality"]
    print(f"  A-Mem | quality     | {_r['correct']:3d}/{_r['total']:3d} = {_r['accuracy_pct']:5.1f}%")
else:
    print("[missing] amem quality -- run the evaluation loop first")
    amem_scores["quality"] = None

# NarrativeQA
_path = RESULTS_DIR / "amem_narrativeqa_250_results.json"
if _path.exists():
    _rec = json.loads(_path.read_text(encoding="utf-8"))
    amem_scores["narrativeqa"] = score_narrativeqa(_rec)
    _r = amem_scores["narrativeqa"]
    print(f"  A-Mem | narrativeqa | {_r['total']:3d} records, mean token F1 = {_r['mean_token_f1_pct']:5.1f}%")
else:
    print("[missing] amem narrativeqa -- run the evaluation loop first")
    amem_scores["narrativeqa"] = None

# HaluEval
_path = RESULTS_DIR / "amem_halueval_250_results.json"
if _path.exists():
    _rec = json.loads(_path.read_text(encoding="utf-8"))
    amem_scores["halueval"] = score_halueval(_rec)
    _r = amem_scores["halueval"]
    print(f"  A-Mem | halueval    | {_r['correct']:3d}/{_r['total']:3d} = {_r['accuracy_pct']:5.1f}%")
else:
    print("[missing] amem halueval -- run the evaluation loop first")
    amem_scores["halueval"] = None


---
## 4 — MemTree

**Description:** MemTree organises document chunks into a hierarchical semantic tree. Leaf nodes hold raw chunks; internal nodes contain LLM-generated summaries of their children. At query time the tree is traversed top-down: at each node the system decides whether to descend into children or use the current summary, allowing coarse-to-fine retrieval.

**Python env:** Python 3.13 required (`sentence-transformers`, model `all-MiniLM-L6-v2`).

**Key parameters:**
- `BASE_THRESHOLD = 0.4` — cosine similarity threshold for tree insertion
- `RATE = 0.5` — threshold decay per tree level
- `MAX_DEPTH = 15` — maximum tree depth
- `TOP_K_RETRIEVE = 3` — top-K nodes returned at query time

**Chunking:** 500 words per chunk for QuALITY and NarrativeQA; 300 words for HaluEval (same as A-Mem).

**Result files:** `../results/group2/memtree_{dataset}_250_results.json`


In [ ]:
# %pip install sentence-transformers openai  (use Python 3.13)

# ---- MemTree system setup -----------------------------------------------
# MemTree requires the MemTree repo to be available. Set MEMTREE_ROOT to the
# repo directory; the code below adds it to sys.path.

MEMTREE_ROOT = os.environ.get("MEMTREE_ROOT", "")  # e.g. C:/path/to/MemTree
if MEMTREE_ROOT and pathlib.Path(MEMTREE_ROOT).exists():
    sys.path.insert(0, MEMTREE_ROOT)
    print(f"[ok] MemTree root added: {MEMTREE_ROOT}")
else:
    print("[warn] MEMTREE_ROOT not set or not found; MemTree import may fail")

try:
    from memtree import MemTree  # type: ignore
    print("[ok] MemTree imported")
except ImportError as e:
    print(f"[error] Could not import MemTree: {e}")

# MemTree hyper-parameters (from paper / repo defaults)
MEMTREE_BASE_THRESHOLD = 0.4
MEMTREE_RATE           = 0.5
MEMTREE_MAX_DEPTH      = 15
MEMTREE_TOP_K          = 3

MEMTREE_CHUNK_SIZES = {
    "quality"    : 500,
    "narrativeqa": 500,
    "halueval"   : 300,
}

import openai as _oai_mt
_mt_client = _oai_mt.OpenAI(api_key=OPENAI_API_KEY)
MEMTREE_MODEL = "gpt-4o-mini"


def _mt_chat(messages: List[Dict], max_tokens: int = 512) -> str:
    """OpenAI chat completion for MemTree answer generation."""
    resp = _mt_client.chat.completions.create(
        model=MEMTREE_MODEL, messages=messages,
        max_tokens=max_tokens, temperature=0.0,
    )
    return resp.choices[0].message.content.strip()


def memtree_answer_question(
    text: str,
    question: str,
    chunk_size: int,
    system_prompt: str = "Answer the question based on the retrieved context.",
) -> Tuple[str, int]:
    """
    Build a MemTree from text chunks, retrieve top-K nodes, then answer.
    Returns (model_answer, num_chunks).
    """
    chunks = chunk_text(text, chunk_size)  # reuse chunk_text from A-Mem cell
    tree = MemTree(
        base_threshold=MEMTREE_BASE_THRESHOLD,
        rate=MEMTREE_RATE,
        max_depth=MEMTREE_MAX_DEPTH,
    )
    for chunk in chunks:
        tree.insert(chunk)
    retrieved = tree.retrieve(question, top_k=MEMTREE_TOP_K)
    context = "\n\n".join(retrieved) if retrieved else (chunks[0] if chunks else "")
    msg = [
        {"role": "system", "content": system_prompt},
        {"role": "user",   "content": f"Context:\n{context}\n\nQuestion: {question}"},
    ]
    answer = _mt_chat(msg)
    return answer, len(chunks)


print("[ok] MemTree helpers defined.")


In [ ]:
# ---- MemTree evaluation loop --------------------------------------------
# Resume-safe; saves every 10 questions.

def run_memtree_quality(force: bool = False):
    out_path = RESULTS_DIR / "memtree_quality_250_results.json"
    results: List[Dict] = []
    done_ids: set = set()
    if out_path.exists() and not force:
        results  = json.loads(out_path.read_text(encoding="utf-8"))
        done_ids = {r["question_unique_id"] for r in results}
        print(f"[resume] quality: {len(done_ids)} already done")
    data      = load_dataset("quality")
    remaining = [q for q in data if q["question_unique_id"] not in done_ids]
    if not remaining:
        print("[skip] quality: already complete"); return
    print(f"[run] quality: {len(remaining)} questions remaining ...")
    chunk_size = MEMTREE_CHUNK_SIZES["quality"]
    for i, item in enumerate(remaining, 1):
        qid = item["question_unique_id"]
        try:
            opts_str = "\n".join(f"{chr(64+j+1)}. {o}" for j, o in enumerate(item["options"]))
            system_p = ("Answer the multiple-choice question using the retrieved context. "
                        "Reply with ONLY the letter A, B, C, or D.")
            full_q   = f"{item['question']}\n\nOptions:\n{opts_str}"
            raw_answer, num_chunks = memtree_answer_question(
                item["article"], full_q, chunk_size, system_prompt=system_p)
            choice_int = {"A":1,"B":2,"C":3,"D":4}.get(raw_answer.strip().upper()[:1], 0)
            gold = int(item["gold_label"])
            is_correct = (choice_int == gold)
        except Exception as e:
            print(f"  [err] {qid}: {e}")
            raw_answer, choice_int, num_chunks = f"[error] {e}", 0, 0
            gold = int(item.get("gold_label", 0))
            is_correct = False
        results.append({
            "question_unique_id": qid,
            "article_id"        : item.get("article_id", ""),
            "title"             : item.get("title", ""),
            "question"          : item["question"],
            "options"           : item["options"],
            "gold_label"        : gold,
            "model_choice"      : choice_int,
            "is_correct"        : is_correct,
            "raw_response"      : raw_answer,
            "num_chunks"        : num_chunks,
        })
        if i % 10 == 0 or i == len(remaining):
            out_path.write_text(json.dumps(results, indent=2, ensure_ascii=False), encoding="utf-8")
            print(f"  [{i}/{len(remaining)}] saved")
    print(f"[done] quality -> {out_path}")


def run_memtree_narrativeqa(force: bool = False):
    out_path = RESULTS_DIR / "memtree_narrativeqa_250_results.json"
    results: List[Dict] = []
    done_ids: set = set()
    if out_path.exists() and not force:
        results  = json.loads(out_path.read_text(encoding="utf-8"))
        done_ids = {r["document_id"] for r in results}
        print(f"[resume] narrativeqa: {len(done_ids)} already done")
    data      = load_dataset("narrativeqa")
    remaining = [q for q in data if q["document_id"] not in done_ids]
    if not remaining:
        print("[skip] narrativeqa: already complete"); return
    print(f"[run] narrativeqa: {len(remaining)} questions remaining ...")
    chunk_size = MEMTREE_CHUNK_SIZES["narrativeqa"]
    for i, item in enumerate(remaining, 1):
        did = item["document_id"]
        try:
            model_answer, num_chunks = memtree_answer_question(
                item["summary"], item["question"], chunk_size)
        except Exception as e:
            print(f"  [err] {did}: {e}")
            model_answer, num_chunks = f"[error] {e}", 0
        results.append({
            "document_id"      : did,
            "question"         : item["question"],
            "reference_answer1": item["answer1"],
            "reference_answer2": item["answer2"],
            "model_answer"     : model_answer,
            "num_chunks"       : num_chunks,
        })
        if i % 10 == 0 or i == len(remaining):
            out_path.write_text(json.dumps(results, indent=2, ensure_ascii=False), encoding="utf-8")
            print(f"  [{i}/{len(remaining)}] saved")
    print(f"[done] narrativeqa -> {out_path}")


def run_memtree_halueval(force: bool = False):
    out_path = RESULTS_DIR / "memtree_halueval_250_results.json"
    results: List[Dict] = []
    done_qs: set = set()
    if out_path.exists() and not force:
        results = json.loads(out_path.read_text(encoding="utf-8"))
        done_qs = {r["question"] for r in results}
        print(f"[resume] halueval: {len(done_qs)} already done")
    data      = load_dataset("halueval")
    remaining = [q for q in data if q["question"] not in done_qs]
    if not remaining:
        print("[skip] halueval: already complete"); return
    print(f"[run] halueval: {len(remaining)} questions remaining ...")
    chunk_size = MEMTREE_CHUNK_SIZES["halueval"]
    for i, item in enumerate(remaining, 1):
        question = item["question"]
        try:
            model_answer, num_chunks = memtree_answer_question(
                item["knowledge"], question, chunk_size)
        except Exception as e:
            print(f"  [err] q{i}: {e}")
            model_answer, num_chunks = f"[error] {e}", 0
        results.append({
            "question"            : question,
            "knowledge"           : item["knowledge"],
            "right_answer"        : item["right_answer"],
            "hallucinated_answer" : item["hallucinated_answer"],
            "model_answer"        : model_answer,
            "num_chunks"          : num_chunks,
        })
        if i % 10 == 0 or i == len(remaining):
            out_path.write_text(json.dumps(results, indent=2, ensure_ascii=False), encoding="utf-8")
            print(f"  [{i}/{len(remaining)}] saved")
    print(f"[done] halueval -> {out_path}")


# Un-comment to run:
# run_memtree_quality()
# run_memtree_narrativeqa()
# run_memtree_halueval()


In [ ]:
# ---- MemTree: load results & compute scores -----------------------------
memtree_scores: Dict[str, Any] = {}

# QuALITY
_path = RESULTS_DIR / "memtree_quality_250_results.json"
if _path.exists():
    _rec = json.loads(_path.read_text(encoding="utf-8"))
    memtree_scores["quality"] = accuracy_mcq(_rec, pred_key="model_choice", gold_key="gold_label")
    _r = memtree_scores["quality"]
    print(f"  MemTree | quality     | {_r['correct']:3d}/{_r['total']:3d} = {_r['accuracy_pct']:5.1f}%")
else:
    print("[missing] memtree quality -- run the evaluation loop first")
    memtree_scores["quality"] = None

# NarrativeQA
_path = RESULTS_DIR / "memtree_narrativeqa_250_results.json"
if _path.exists():
    _rec = json.loads(_path.read_text(encoding="utf-8"))
    memtree_scores["narrativeqa"] = score_narrativeqa(_rec)
    _r = memtree_scores["narrativeqa"]
    print(f"  MemTree | narrativeqa | {_r['total']:3d} records, mean token F1 = {_r['mean_token_f1_pct']:5.1f}%")
else:
    print("[missing] memtree narrativeqa -- run the evaluation loop first")
    memtree_scores["narrativeqa"] = None

# HaluEval
_path = RESULTS_DIR / "memtree_halueval_250_results.json"
if _path.exists():
    _rec = json.loads(_path.read_text(encoding="utf-8"))
    memtree_scores["halueval"] = score_halueval(_rec)
    _r = memtree_scores["halueval"]
    print(f"  MemTree | halueval    | {_r['correct']:3d}/{_r['total']:3d} = {_r['accuracy_pct']:5.1f}%")
else:
    print("[missing] memtree halueval -- run the evaluation loop first")
    memtree_scores["halueval"] = None


---
## 5 — MemGPT / Letta

**Description:** MemGPT (Packer et al., 2023) implements a tiered memory OS with working memory (context window), recall memory (searchable conversation store), and archival memory (external vector store). It uses function-calling to manage its own context, self-evicting old information when the context fills.

**GitHub:** [cpacker/MemGPT](https://github.com/cpacker/MemGPT)

**Python env:** Python 3.10+ (no `sentence-transformers` dependency for Group 2 evaluation).

**Install:** `pip install letta`

**Architecture for long documents:**
- Document text is split into chunks and inserted into the agent's archival memory via `archival_memory_insert`
- At query time the agent uses `archival_memory_search` to retrieve relevant chunks
- A fresh agent is created per question to avoid memory contamination across documents

**IMPORTANT — scoring note:**
The stored `is_correct` field in Quality results has a type-mismatch bug (comparing int to string). Always **recompute** accuracy by mapping `model_letter` (A→1, B→2, C→3, D→4) against `gold_label` (int). For NarrativeQA the pre-computed `f1` field (string float) is reliable. For HaluEval `is_correct` is stored as string `'0'`/`'1'`.

**Result files:** `../results/group2/memgpt_{dataset}_250_results.json`


In [ ]:
# %pip install letta

# ---- MemGPT / Letta system setup ----------------------------------------
# MemGPT uses the Letta library. Each document gets a fresh agent instance
# to prevent cross-document memory contamination.

try:
    from letta import create_client  # type: ignore
    from letta.schemas.memory import ChatMemory  # type: ignore
    print("[ok] Letta client imported")
except ImportError as e:
    print(f"[error] Could not import Letta: {e}\n       Run: pip install letta")

MEMGPT_MODEL      = "gpt-4o-mini"
MEMGPT_CHUNK_SIZE = 500  # words per archival memory chunk

# Letta client (connects to local Letta server or cloud)
# Set LETTA_BASE_URL env var for a custom server, or leave default for local.
_letta_client = None

def get_letta_client():
    """Lazily initialise the Letta client."""
    global _letta_client
    if _letta_client is None:
        _letta_client = create_client()
    return _letta_client


def _make_memgpt_agent(persona: str = "You are a helpful assistant that answers questions from stored memories."):
    """Create a fresh MemGPT agent with empty archival memory."""
    client = get_letta_client()
    agent_state = client.create_agent(
        memory=ChatMemory(human="User", persona=persona),
        llm_config=None,  # uses server default; override if needed
    )
    return agent_state.id


def memgpt_insert_chunks(agent_id: str, text: str, chunk_size: int = MEMGPT_CHUNK_SIZE):
    """Insert all text chunks into archival memory."""
    client = get_letta_client()
    words  = text.split()
    for i in range(0, len(words), chunk_size):
        chunk = " ".join(words[i:i+chunk_size])
        client.insert_archival_memory(agent_id, memory=chunk)


def memgpt_ask(agent_id: str, question: str) -> str:
    """Send a question to the MemGPT agent and return its final response."""
    client = get_letta_client()
    response = client.send_message(agent_id=agent_id, message=question, role="user")
    # Extract the last assistant message
    for msg in reversed(response.messages):
        if hasattr(msg, "text") and msg.text:
            return msg.text.strip()
        if hasattr(msg, "assistant_message") and msg.assistant_message:
            return msg.assistant_message.strip()
    return ""


def memgpt_delete_agent(agent_id: str):
    """Clean up agent after use."""
    try:
        get_letta_client().delete_agent(agent_id)
    except Exception:
        pass


print("[ok] MemGPT helpers defined.")


In [ ]:
# ---- MemGPT evaluation loop ---------------------------------------------

def run_memgpt_quality(force: bool = False):
    out_path = RESULTS_DIR / "memgpt_quality_250_results.json"
    results: List[Dict] = []
    done_ids: set = set()
    if out_path.exists() and not force:
        results  = json.loads(out_path.read_text(encoding="utf-8"))
        done_ids = {r["question_unique_id"] for r in results}
        print(f"[resume] quality: {len(done_ids)} already done")
    data      = load_dataset("quality")
    remaining = [q for q in data if q["question_unique_id"] not in done_ids]
    if not remaining:
        print("[skip] quality: already complete"); return
    print(f"[run] quality: {len(remaining)} questions remaining ...")
    for i, item in enumerate(remaining, 1):
        qid = item["question_unique_id"]
        agent_id = None
        try:
            agent_id = _make_memgpt_agent()
            memgpt_insert_chunks(agent_id, item["article"])
            opts_str = "\n".join(f"{chr(64+j+1)}. {o}" for j, o in enumerate(item["options"]))
            prompt   = (f"Answer this multiple-choice question using your archival memories. "
                        f"Reply with ONLY the letter A, B, C, or D.\n\n"
                        f"Question: {item['question']}\n\nOptions:\n{opts_str}")
            raw_answer = memgpt_ask(agent_id, prompt)
            model_letter = raw_answer.strip().upper()[:1] if raw_answer else ""
            choice_int   = {"A":1,"B":2,"C":3,"D":4}.get(model_letter, 0)
            gold = int(item["gold_label"])
            is_correct = (choice_int == gold)
        except Exception as e:
            print(f"  [err] {qid}: {e}")
            raw_answer, model_letter, choice_int = f"[error] {e}", "", 0
            gold = int(item.get("gold_label", 0))
            is_correct = False
        finally:
            if agent_id:
                memgpt_delete_agent(agent_id)
        results.append({
            "question_unique_id": qid,
            "title"             : item.get("title", ""),
            "question"          : item["question"],
            "gold_label"        : gold,
            "model_answer"      : raw_answer,
            "model_letter"      : model_letter,
            "is_correct"        : is_correct,
        })
        if i % 10 == 0 or i == len(remaining):
            out_path.write_text(json.dumps(results, indent=2, ensure_ascii=False), encoding="utf-8")
            print(f"  [{i}/{len(remaining)}] saved")
    print(f"[done] quality -> {out_path}")


def run_memgpt_narrativeqa(force: bool = False):
    out_path = RESULTS_DIR / "memgpt_narrativeqa_250_results.json"
    results: List[Dict] = []
    done_ids: set = set()
    if out_path.exists() and not force:
        results  = json.loads(out_path.read_text(encoding="utf-8"))
        done_ids = {r["document_id"] for r in results}
        print(f"[resume] narrativeqa: {len(done_ids)} already done")
    data      = load_dataset("narrativeqa")
    remaining = [q for q in data if q["document_id"] not in done_ids]
    if not remaining:
        print("[skip] narrativeqa: already complete"); return
    print(f"[run] narrativeqa: {len(remaining)} questions remaining ...")
    for i, item in enumerate(remaining, 1):
        did = item["document_id"]
        agent_id = None
        try:
            agent_id = _make_memgpt_agent()
            memgpt_insert_chunks(agent_id, item["summary"])
            model_answer = memgpt_ask(agent_id, f"Answer concisely: {item['question']}")
            f1_score = token_f1_best(model_answer, item["answer1"], item["answer2"])
        except Exception as e:
            print(f"  [err] {did}: {e}")
            model_answer, f1_score = f"[error] {e}", 0.0
        finally:
            if agent_id:
                memgpt_delete_agent(agent_id)
        results.append({
            "document_id" : did,
            "question"    : item["question"],
            "answer1"     : item["answer1"],
            "answer2"     : item["answer2"],
            "model_answer": model_answer,
            "f1"          : str(round(f1_score, 4)),
            "is_correct"  : "1" if f1_score > 0 else "0",
        })
        if i % 10 == 0 or i == len(remaining):
            out_path.write_text(json.dumps(results, indent=2, ensure_ascii=False), encoding="utf-8")
            print(f"  [{i}/{len(remaining)}] saved")
    print(f"[done] narrativeqa -> {out_path}")


def run_memgpt_halueval(force: bool = False):
    out_path = RESULTS_DIR / "memgpt_halueval_250_results.json"
    results: List[Dict] = []
    done_qs: set = set()
    if out_path.exists() and not force:
        results = json.loads(out_path.read_text(encoding="utf-8"))
        done_qs = {r["question"] for r in results}
        print(f"[resume] halueval: {len(done_qs)} already done")
    data      = load_dataset("halueval")
    remaining = [q for q in data if q["question"] not in done_qs]
    if not remaining:
        print("[skip] halueval: already complete"); return
    print(f"[run] halueval: {len(remaining)} questions remaining ...")
    for i, item in enumerate(remaining, 1):
        question = item["question"]
        agent_id = None
        try:
            agent_id = _make_memgpt_agent()
            memgpt_insert_chunks(agent_id, item["knowledge"], chunk_size=300)
            model_answer = memgpt_ask(agent_id, f"Answer concisely: {question}")
            is_correct   = "1" if contains_match(model_answer, item["right_answer"]) else "0"
        except Exception as e:
            print(f"  [err] q{i}: {e}")
            model_answer, is_correct = f"[error] {e}", "0"
        finally:
            if agent_id:
                memgpt_delete_agent(agent_id)
        results.append({
            "question"            : question,
            "knowledge"           : item["knowledge"],
            "right_answer"        : item["right_answer"],
            "hallucinated_answer" : item["hallucinated_answer"],
            "model_answer"        : model_answer,
            "is_correct"          : is_correct,
        })
        if i % 10 == 0 or i == len(remaining):
            out_path.write_text(json.dumps(results, indent=2, ensure_ascii=False), encoding="utf-8")
            print(f"  [{i}/{len(remaining)}] saved")
    print(f"[done] halueval -> {out_path}")


# Un-comment to run:
# run_memgpt_quality()
# run_memgpt_narrativeqa()
# run_memgpt_halueval()


In [ ]:
# ---- MemGPT: load results & compute scores ------------------------------
# IMPORTANT: always recompute quality accuracy from model_letter (not is_correct)
# because the stored is_correct has a type-mismatch bug (int vs string comparison).
# NarrativeQA f1 field is a pre-computed string float — use it directly.
# HaluEval is_correct is a string '0'/'1'.

memgpt_scores: Dict[str, Any] = {}

# QuALITY — recompute from model_letter vs gold_label
_path = RESULTS_DIR / "memgpt_quality_250_results.json"
if _path.exists():
    _rec = json.loads(_path.read_text(encoding="utf-8"))
    memgpt_scores["quality"] = accuracy_mcq(_rec, pred_key="model_letter", gold_key="gold_label")
    _r = memgpt_scores["quality"]
    print(f"  MemGPT | quality     | {_r['correct']:3d}/{_r['total']:3d} = {_r['accuracy_pct']:5.1f}%")
else:
    print("[missing] memgpt quality -- run the evaluation loop first")
    memgpt_scores["quality"] = None

# NarrativeQA — mean of pre-computed f1 strings
_path = RESULTS_DIR / "memgpt_narrativeqa_250_results.json"
if _path.exists():
    _rec = json.loads(_path.read_text(encoding="utf-8"))
    _f1s = [float(r["f1"]) for r in _rec if r.get("f1") not in (None, "")]
    mean_f1 = round(100 * sum(_f1s) / len(_f1s), 1) if _f1s else 0.0
    memgpt_scores["narrativeqa"] = {"total": len(_f1s), "mean_token_f1_pct": mean_f1}
    _r = memgpt_scores["narrativeqa"]
    print(f"  MemGPT | narrativeqa | {_r['total']:3d} records, mean token F1 = {_r['mean_token_f1_pct']:5.1f}%")
else:
    print("[missing] memgpt narrativeqa -- run the evaluation loop first")
    memgpt_scores["narrativeqa"] = None

# HaluEval — is_correct is string '0'/'1'
_path = RESULTS_DIR / "memgpt_halueval_250_results.json"
if _path.exists():
    _rec = json.loads(_path.read_text(encoding="utf-8"))
    _correct = sum(1 for r in _rec if str(r.get("is_correct", "0")) == "1")
    _total   = len(_rec)
    _acc     = round(100 * _correct / _total, 1) if _total > 0 else 0.0
    memgpt_scores["halueval"] = {"correct": _correct, "total": _total, "accuracy_pct": _acc}
    _r = memgpt_scores["halueval"]
    print(f"  MemGPT | halueval    | {_r['correct']:3d}/{_r['total']:3d} = {_r['accuracy_pct']:5.1f}%")
else:
    print("[missing] memgpt halueval -- run the evaluation loop first")
    memgpt_scores["halueval"] = None


---
## 6 — MemOS

**Description:** MemOS uses a cloud-hosted structured memory API to extract, store, and retrieve memories from raw text. Given a document, MemOS extracts structured memory items via LLM, stores them in a cloud memory store, then retrieves relevant items to answer queries.

**Cloud API:** `https://memos.memtensor.cn/api/openmem/v1`
**Auth:** `Token {MEMOS_API_KEY}` (set `MEMOS_API_KEY` in `.env`)
**Python env:** Python 3.13.

**Performance note:** Memory extraction takes ~100 seconds per QuALITY article (LLM-based extraction from ~30K char documents). For NarrativeQA summaries (~2K chars) and HaluEval passages (~350 chars) it is much faster.

**Per-call flow:**
1. Create a memory store for the document
2. Add document text (MemOS extracts structured memories server-side)
3. Query with the question to retrieve relevant memories
4. Generate answer with OpenAI using retrieved memories as context
5. Delete the memory store

**Result files:** `../results/group2/memos_{dataset}_250_results.json`


In [ ]:
# ---- MemOS system setup -------------------------------------------------
import urllib.request
import urllib.error

MEMOS_BASE_URL = "https://memos.memtensor.cn/api/openmem/v1"

if not MEMOS_API_KEY:
    print("[warn] MEMOS_API_KEY not set — MemOS evaluation will fail")
else:
    print(f"[ok] MEMOS_API_KEY loaded (prefix: {MEMOS_API_KEY[:6]}...)")

import openai as _oai_memos
_memos_oai_client = _oai_memos.OpenAI(api_key=OPENAI_API_KEY)
MEMOS_LLM_MODEL = "gpt-4o-mini"


def _memos_request(method: str, endpoint: str, payload: Optional[Dict] = None,
                   retries: int = 5, retry_delay: float = 5.0) -> Dict:
    """Make an authenticated request to the MemOS cloud API with retry on timeout."""
    import json as _json
    import urllib.request, urllib.error, ssl
    url = f"{MEMOS_BASE_URL}{endpoint}"
    headers = {
        "Authorization": f"Token {MEMOS_API_KEY}",
        "Content-Type" : "application/json",
    }
    data = _json.dumps(payload).encode("utf-8") if payload else None
    for attempt in range(1, retries + 1):
        try:
            req = urllib.request.Request(url, data=data, headers=headers, method=method)
            ctx = ssl.create_default_context()
            with urllib.request.urlopen(req, context=ctx, timeout=120) as resp:
                return _json.loads(resp.read().decode("utf-8"))
        except urllib.error.HTTPError as e:
            raise
        except (urllib.error.URLError, TimeoutError) as e:
            if attempt < retries:
                print(f"  [retry {attempt}/{retries}] {e}; sleeping {retry_delay}s ...")
                time.sleep(retry_delay)
            else:
                raise


def memos_create_store(name: str) -> str:
    """Create a new MemOS memory store and return its ID."""
    resp = _memos_request("POST", "/memory/stores/", {"name": name, "description": ""})
    return resp["id"]


def memos_add_text(store_id: str, text: str):
    """Add raw text to a MemOS memory store (server extracts structured memories)."""
    _memos_request("POST", f"/memory/stores/{store_id}/add/",
                   {"text": text, "extract": True})


def memos_search(store_id: str, query: str, top_k: int = 5) -> List[str]:
    """Retrieve top-K relevant memory items from a MemOS store."""
    resp = _memos_request("POST", f"/memory/stores/{store_id}/search/",
                          {"query": query, "top_k": top_k})
    memories = resp.get("results", resp.get("memories", []))
    texts = []
    for m in memories:
        if isinstance(m, str):
            texts.append(m)
        elif isinstance(m, dict):
            texts.append(m.get("content", m.get("text", str(m))))
    return texts


def memos_delete_store(store_id: str):
    """Delete a MemOS memory store."""
    try:
        _memos_request("DELETE", f"/memory/stores/{store_id}/")
    except Exception:
        pass


def memos_answer_question(text: str, question: str,
                          system_prompt: str = "Answer the question based on the retrieved memories.",
                          top_k: int = 5) -> Tuple[str, int]:
    """
    Full MemOS pipeline: create store → add text → search → answer → delete.
    Returns (model_answer, num_memories_retrieved).
    """
    import uuid
    store_id = memos_create_store(f"eval_{uuid.uuid4().hex[:8]}")
    try:
        memos_add_text(store_id, text)
        memories = memos_search(store_id, question, top_k=top_k)
        context  = "\n".join(f"- {m}" for m in memories) if memories else text[:2000]
        msg = [
            {"role": "system", "content": system_prompt},
            {"role": "user",   "content": f"Retrieved memories:\n{context}\n\nQuestion: {question}"},
        ]
        resp = _memos_oai_client.chat.completions.create(
            model=MEMOS_LLM_MODEL, messages=msg, max_tokens=512, temperature=0.0,
        )
        answer = resp.choices[0].message.content.strip()
    finally:
        memos_delete_store(store_id)
    return answer, len(memories)


print("[ok] MemOS helpers defined.")


In [ ]:
# ---- MemOS evaluation loop ----------------------------------------------
# NOTE: Quality articles are ~30K chars; MemOS extraction takes ~100s each.
# At 250 questions this run will take ~7 hours. Resume-safe.

def run_memos_quality(force: bool = False):
    out_path = RESULTS_DIR / "memos_quality_250_results.json"
    results: List[Dict] = []
    done_ids: set = set()
    if out_path.exists() and not force:
        results  = json.loads(out_path.read_text(encoding="utf-8"))
        done_ids = {r["question_unique_id"] for r in results}
        print(f"[resume] quality: {len(done_ids)} already done")
    data      = load_dataset("quality")
    remaining = [q for q in data if q["question_unique_id"] not in done_ids]
    if not remaining:
        print("[skip] quality: already complete"); return
    print(f"[run] quality: {len(remaining)} questions remaining (est. ~{len(remaining)*100//3600}h) ...")
    letter_map = {1:"A", 2:"B", 3:"C", 4:"D"}
    for i, item in enumerate(remaining, 1):
        qid = item["question_unique_id"]
        gold = int(item["gold_label"])
        gold_letter = letter_map.get(gold, "")
        try:
            opts_str  = "\n".join(f"{chr(64+j+1)}. {o}" for j, o in enumerate(item["options"]))
            system_p  = ("Answer the multiple-choice question using the retrieved memories. "
                         "Reply with ONLY the letter A, B, C, or D.")
            full_q    = f"{item['question']}\n\nOptions:\n{opts_str}"
            model_answer, n_mem = memos_answer_question(item["article"], full_q, system_prompt=system_p)
            model_letter = model_answer.strip().upper()[:1]
            choice_int   = {"A":1,"B":2,"C":3,"D":4}.get(model_letter, 0)
            is_correct   = str(int(choice_int == gold))
        except Exception as e:
            print(f"  [err] {qid}: {e}")
            model_answer, model_letter, n_mem = f"[error] {e}", "", 0
            choice_int, is_correct = 0, "0"
        results.append({
            "question_unique_id": qid,
            "title"             : item.get("title", ""),
            "question"          : item["question"],
            "gold_label"        : gold,
            "gold_letter"       : gold_letter,
            "model_answer"      : model_answer,
            "model_letter"      : model_letter,
            "memories_retrieved": n_mem,
            "is_correct"        : is_correct,
        })
        if i % 10 == 0 or i == len(remaining):
            out_path.write_text(json.dumps(results, indent=2, ensure_ascii=False), encoding="utf-8")
            print(f"  [{i}/{len(remaining)}] saved")
    print(f"[done] quality -> {out_path}")


def run_memos_narrativeqa(force: bool = False):
    out_path = RESULTS_DIR / "memos_narrativeqa_250_results.json"
    results: List[Dict] = []
    done_ids: set = set()
    if out_path.exists() and not force:
        results  = json.loads(out_path.read_text(encoding="utf-8"))
        done_ids = {r["document_id"] for r in results}
        print(f"[resume] narrativeqa: {len(done_ids)} already done")
    data      = load_dataset("narrativeqa")
    remaining = [q for q in data if q["document_id"] not in done_ids]
    if not remaining:
        print("[skip] narrativeqa: already complete"); return
    print(f"[run] narrativeqa: {len(remaining)} questions remaining ...")
    for i, item in enumerate(remaining, 1):
        did = item["document_id"]
        try:
            model_answer, n_mem = memos_answer_question(item["summary"], item["question"])
            f1_score = token_f1_best(model_answer, item["answer1"], item["answer2"])
            is_correct = str(int(f1_score > 0))
        except Exception as e:
            print(f"  [err] {did}: {e}")
            model_answer, n_mem, f1_score = f"[error] {e}", 0, 0.0
            is_correct = "0"
        results.append({
            "document_id"      : did,
            "question"         : item["question"],
            "answer1"          : item["answer1"],
            "answer2"          : item["answer2"],
            "model_answer"     : model_answer,
            "memories_retrieved": n_mem,
            "f1"               : str(round(f1_score, 4)),
            "is_correct"       : is_correct,
        })
        if i % 10 == 0 or i == len(remaining):
            out_path.write_text(json.dumps(results, indent=2, ensure_ascii=False), encoding="utf-8")
            print(f"  [{i}/{len(remaining)}] saved")
    print(f"[done] narrativeqa -> {out_path}")


def run_memos_halueval(force: bool = False):
    out_path = RESULTS_DIR / "memos_halueval_250_results.json"
    results: List[Dict] = []
    done_qs: set = set()
    if out_path.exists() and not force:
        results = json.loads(out_path.read_text(encoding="utf-8"))
        done_qs = {r["question"] for r in results}
        print(f"[resume] halueval: {len(done_qs)} already done")
    data      = load_dataset("halueval")
    remaining = [q for q in data if q["question"] not in done_qs]
    if not remaining:
        print("[skip] halueval: already complete"); return
    print(f"[run] halueval: {len(remaining)} questions remaining ...")
    for i, item in enumerate(remaining, 1):
        question = item["question"]
        try:
            model_answer, n_mem = memos_answer_question(item["knowledge"], question)
            is_correct = str(int(contains_match(model_answer, item["right_answer"])))
        except Exception as e:
            print(f"  [err] q{i}: {e}")
            model_answer, n_mem, is_correct = f"[error] {e}", 0, "0"
        results.append({
            "question"            : question,
            "knowledge"           : item["knowledge"],
            "right_answer"        : item["right_answer"],
            "hallucinated_answer" : item["hallucinated_answer"],
            "model_answer"        : model_answer,
            "memories_retrieved"  : n_mem,
            "is_correct"          : is_correct,
        })
        if i % 10 == 0 or i == len(remaining):
            out_path.write_text(json.dumps(results, indent=2, ensure_ascii=False), encoding="utf-8")
            print(f"  [{i}/{len(remaining)}] saved")
    print(f"[done] halueval -> {out_path}")


# Un-comment to run:
# run_memos_quality()
# run_memos_narrativeqa()
# run_memos_halueval()


In [ ]:
# ---- MemOS: load results & compute scores -------------------------------
# is_correct is stored as string for all three datasets.
# NarrativeQA f1 is a pre-computed string float.

memos_scores: Dict[str, Any] = {}

# QuALITY — is_correct string '0'/'1', but also recompute from model_letter for safety
_path = RESULTS_DIR / "memos_quality_250_results.json"
if _path.exists():
    _rec = json.loads(_path.read_text(encoding="utf-8"))
    memos_scores["quality"] = accuracy_mcq(_rec, pred_key="model_letter", gold_key="gold_label")
    _r = memos_scores["quality"]
    print(f"  MemOS | quality     | {_r['correct']:3d}/{_r['total']:3d} = {_r['accuracy_pct']:5.1f}%")
else:
    print("[missing] memos quality -- run the evaluation loop first")
    memos_scores["quality"] = None

# NarrativeQA — mean of pre-computed f1 strings
_path = RESULTS_DIR / "memos_narrativeqa_250_results.json"
if _path.exists():
    _rec = json.loads(_path.read_text(encoding="utf-8"))
    _f1s = [float(r["f1"]) for r in _rec if r.get("f1") not in (None, "")]
    mean_f1 = round(100 * sum(_f1s) / len(_f1s), 1) if _f1s else 0.0
    memos_scores["narrativeqa"] = {"total": len(_f1s), "mean_token_f1_pct": mean_f1}
    _r = memos_scores["narrativeqa"]
    print(f"  MemOS | narrativeqa | {_r['total']:3d} records, mean token F1 = {_r['mean_token_f1_pct']:5.1f}%")
else:
    print("[missing] memos narrativeqa -- run the evaluation loop first")
    memos_scores["narrativeqa"] = None

# HaluEval — is_correct string '0'/'1'
_path = RESULTS_DIR / "memos_halueval_250_results.json"
if _path.exists():
    _rec = json.loads(_path.read_text(encoding="utf-8"))
    _correct = sum(1 for r in _rec if str(r.get("is_correct", "0")) == "1")
    _total   = len(_rec)
    _acc     = round(100 * _correct / _total, 1) if _total > 0 else 0.0
    memos_scores["halueval"] = {"correct": _correct, "total": _total, "accuracy_pct": _acc}
    _r = memos_scores["halueval"]
    print(f"  MemOS | halueval    | {_r['correct']:3d}/{_r['total']:3d} = {_r['accuracy_pct']:5.1f}%")
else:
    print("[missing] memos halueval -- run the evaluation loop first")
    memos_scores["halueval"] = None


---
## 7 — Summary Table

Aggregate all per-system scores into a single comparison table.

- **QuALITY**: Accuracy % (MCQ, chance = 25%)
- **NarrativeQA**: Mean Token F1 %
- **HaluEval**: Contains-Match %


In [ ]:
# ---- Final summary table ------------------------------------------------

def _get_score(scores: Optional[Dict], dataset: str, metric_key: str) -> str:
    """Extract a score value as a formatted string, or '—' if missing."""
    if scores is None:
        return "—"
    s = scores.get(dataset)
    if s is None:
        return "—"
    val = s.get(metric_key)
    if val is None:
        return "—"
    return f"{val:.1f}%"


all_scores = {
    "ReadAgent": readagent_scores,
    "A-Mem"    : amem_scores,
    "MemTree"  : memtree_scores,
    "MemGPT"   : memgpt_scores,
    "MemOS"    : memos_scores,
}

# Build table rows
rows = []
for system, scores in all_scores.items():
    q_acc   = _get_score(scores, "quality",     "accuracy_pct")
    nqa_f1  = _get_score(scores, "narrativeqa", "mean_token_f1_pct")
    halu_cm = _get_score(scores, "halueval",    "accuracy_pct")

    def _avg(vals):
        nums = [float(v.rstrip("%")) for v in vals if v != "—"]
        return f"{sum(nums)/len(nums):.1f}%" if nums else "—"

    avg = _avg([q_acc, nqa_f1, halu_cm])
    rows.append((system, q_acc, nqa_f1, halu_cm, avg))

# Print as formatted table
header = f"{'System':<12} | {'QuALITY Acc':>11} | {'NarrativeQA F1':>14} | {'HaluEval CM':>11} | {'Avg':>7}"
sep    = "-" * len(header)
print("\nGroup 2 — Long-Document Understanding Results (250 questions each)")
print(sep)
print(header)
print(sep)
for system, q, n, h, avg in rows:
    print(f"{system:<12} | {q:>11} | {n:>14} | {h:>11} | {avg:>7}")
print(sep)
print("\nMetrics:")
print("  QuALITY Acc   : MCQ accuracy (chance = 25%)")
print("  NarrativeQA F1: Mean token-overlap F1")
print("  HaluEval CM   : Contains-match (right_answer in model_answer)")
print("  Avg           : Simple mean of the three metrics")


In [ ]:
# ---- Optional: render as pandas DataFrame (nicer in Jupyter) -----------
try:
    import pandas as pd
    df = pd.DataFrame(rows, columns=["System", "QuALITY Acc", "NarrativeQA F1", "HaluEval CM", "Avg"])
    df = df.set_index("System")
    display(df)
except ImportError:
    print("[info] pandas not installed; skipping DataFrame display")
except NameError:
    print("[info] display() not available outside Jupyter")
